<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-10-tuning-and-evaluation/lesson-10.2-context-caching/notebooks/GCP_Capstone_10.2_ContextCaching.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10.2 Context Caching — 90% Savings
**Netsetos GenAI Engineering — GCP Capstone**


In [ ]:
!pip install -q google-genai

from google.colab import auth
auth.authenticate_user()

from google import genai
from google.genai.types import (
    Content, Part, CreateCachedContentConfig,
    UpdateCachedContentConfig, GenerateContentConfig
)
import hashlib, time

PROJECT = 'your-project-id'
LOCATION = 'us-central1'  # Regional endpoint for consistent cache hits

client = genai.Client(enterprise=True, project=PROJECT, location=LOCATION)
print('SDK ready')


## Cell 1: Verify Implicit Cache Hit


In [ ]:
# Implicit caching is automatic. Just check usage_metadata.
# Use a long STABLE prefix + variable short suffix.

LONG_CONTEXT = '''You are DocuMind AI, a document analysis assistant.
''' + ('Analyze documents carefully. Cite sections. Be concise. ' * 500)  # ~5000 tokens

def make_request(question: str):
    response = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=[LONG_CONTEXT, question]  # Stable first, variable last
    )
    um = response.usage_metadata
    return {
        'question': question,
        'prompt_tokens': um.prompt_token_count,
        'cached_tokens': um.cached_content_token_count or 0,
        'text': response.text[:100]
    }

# First call: no cache hit expected (populates)
r1 = make_request('Summarize in one sentence.')
print(f'Request 1: {r1["cached_tokens"]}/{r1["prompt_tokens"]} cached')

# Second call with SAME prefix: implicit cache hit expected
r2 = make_request('List 3 key points.')
print(f'Request 2: {r2["cached_tokens"]}/{r2["prompt_tokens"]} cached')

# If r2 cached_tokens > 0, you got the 90% discount for free


## Cell 2: Create Explicit Cache


In [ ]:
# Explicit caching: guaranteed hits, full control.
# Template (requires actual GCS bucket with PDF)

def create_explicit_cache(gcs_pdf_uri: str, display_name: str, ttl: str = '7200s'):
    cache = client.caches.create(
        model='gemini-3.6-flash',
        config=CreateCachedContentConfig(
            display_name=display_name,
            system_instruction=(
                'You are DocuMind AI. Answer questions using ONLY '
                'the provided documents. Cite specific section numbers.'
            ),
            contents=[Content(role='user', parts=[
                Part.from_uri(file_uri=gcs_pdf_uri,
                              mime_type='application/pdf')
            ])],
            ttl=ttl,
        ),
    )
    return cache

# Example usage (commented - requires real GCS URI)
# cache = create_explicit_cache('gs://documind/compliance.pdf', 'compliance-v1')
# print(f'Name: {cache.name}')
# print(f'Tokens: {cache.usage_metadata.total_token_count}')
# print(f'Expires: {cache.expire_time}')
print('create_explicit_cache function ready')


## Cell 3: Multi-Turn Chat with Cached Context


In [ ]:
# Chat session with cached compliance doc - each turn gets 90% off
def chat_with_cache(cache_name: str, messages: list[str]):
    chat = client.chats.create(
        model='gemini-3.6-flash',
        config=GenerateContentConfig(cached_content=cache_name),
    )
    results = []
    for msg in messages:
        response = chat.send_message(msg)
        results.append({
            'user': msg,
            'cached_tokens': response.usage_metadata.cached_content_token_count or 0,
            'prompt_tokens': response.usage_metadata.prompt_token_count,
            'response': response.text[:150]
        })
    return results

# Example (commented - requires actual cache)
# messages = [
#     'Summarize the data privacy requirements.',
#     'How do they compare to GDPR Article 17?',
#     'Draft a compliance checklist based on both.'
# ]
# results = chat_with_cache(cache.name, messages)
# for r in results:
#     print(f'{r["cached_tokens"]}/{r["prompt_tokens"]} cached: {r["user"]}')
print('chat_with_cache function ready')


## Cell 4: Cache Lifecycle CRUD


In [ ]:
# Full cache lifecycle: list, get, update TTL, delete with error handling

def list_all_caches():
    caches = []
    for c in client.caches.list():
        caches.append({
            'name': c.name,
            'display_name': c.display_name,
            'tokens': c.usage_metadata.total_token_count if c.usage_metadata else 0,
            'expires': c.expire_time,
        })
    return caches

def extend_cache_ttl(cache_name: str, new_ttl: str):
    return client.caches.update(
        name=cache_name,
        config=UpdateCachedContentConfig(ttl=new_ttl)
    )

def delete_cache_safely(cache_name: str):
    try:
        client.caches.delete(name=cache_name)
        print(f'Deleted: {cache_name}')
    except Exception as e:
        print(f'Delete failed (may be already expired): {e}')

def query_with_cache_retry(cache_name: str, query: str):
    try:
        return client.models.generate_content(
            model='gemini-3.6-flash',
            contents=query,
            config=GenerateContentConfig(cached_content=cache_name),
        )
    except Exception as e:
        if 'not found' in str(e).lower():
            print('Cache expired, caller should recreate')
            raise
        raise

print('CRUD operations defined')
print(f'Current caches: {len(list_all_caches())}')


## Cell 5: RAG Cache Pattern - Cache Corpus, Vary Query


In [ ]:
# DocuMind's primary production pattern
def create_rag_cache(doc_uris: list[str], system_instruction: str,
                     display_name: str, ttl: str = '14400s'):
    '''Cache a multi-document corpus for RAG.'''
    parts = [Part.from_uri(file_uri=uri, mime_type='application/pdf')
             for uri in doc_uris]
    
    cache = client.caches.create(
        model='gemini-3.6-flash',
        config=CreateCachedContentConfig(
            display_name=display_name,
            system_instruction=system_instruction,
            contents=[Content(role='user', parts=parts)],
            ttl=ttl,
        ),
    )
    return cache

def query_rag(cache_name: str, question: str):
    '''Query against cached RAG corpus.'''
    response = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=question,  # Variable: only the user question
        config=GenerateContentConfig(cached_content=cache_name),
    )
    return {
        'answer': response.text,
        'cached_tokens': response.usage_metadata.cached_content_token_count or 0,
        'total_input': response.usage_metadata.prompt_token_count,
        'hit_ratio': (response.usage_metadata.cached_content_token_count or 0) / response.usage_metadata.prompt_token_count if response.usage_metadata.prompt_token_count else 0
    }

print('RAG cache functions ready')


## Cell 6: Break-Even Calculator


In [ ]:
# Does explicit caching pay off for YOUR workload?
def break_even_analysis(
    cache_tokens: int,          # How big is your cache
    queries_per_hour: int,      # How often queried
    session_hours: int = 1,     # Cache lifetime
    model: str = 'gemini-3.6-flash'
):
    prices = {
        'gemini-3.6-flash':      {'std': 1.50, 'cached': 0.15,   'storage': 1.00},
        'gemini-3.1-flash-lite': {'std': 0.25, 'cached': 0.025,   'storage': 1.00},
        'gemini-3.1-pro-preview':        {'std': 2.00, 'cached': 0.20,  'storage': 4.50},
    }
    p = prices.get(model, prices['gemini-3.6-flash'])
    
    tokens_m = cache_tokens / 1_000_000
    total_queries = queries_per_hour * session_hours
    
    no_cache_cost = total_queries * tokens_m * p['std']
    
    cache_creation = tokens_m * p['std']
    cached_reads = total_queries * tokens_m * p['cached']
    storage = session_hours * tokens_m * p['storage']
    with_cache_cost = cache_creation + cached_reads + storage
    
    savings = no_cache_cost - with_cache_cost
    savings_pct = (savings / no_cache_cost * 100) if no_cache_cost > 0 else 0
    
    break_even_qph = (tokens_m * p['storage']) / (tokens_m * (p['std'] - p['cached']))
    
    return {
        'no_cache_cost': round(no_cache_cost, 2),
        'with_cache_cost': round(with_cache_cost, 2),
        'savings': round(savings, 2),
        'savings_pct': round(savings_pct, 1),
        'break_even_qph': round(break_even_qph, 1),
        'worth_it': savings > 0
    }

# DocuMind scenarios
print('DocuMind break-even scenarios (Gemini 3.6 Flash):')
print('-' * 65)
for scenario, params in [
    ('Low volume: 2 qph, 1 hr, 100K cache', (100_000, 2, 1)),
    ('Medium: 10 qph, 4 hr, 500K cache',    (500_000, 10, 4)),
    ('High: 100 qph, 8 hr, 1M cache',        (1_000_000, 100, 8)),
    ('Enterprise: 500 qph, 8 hr, 2M cache',  (2_000_000, 500, 8)),
]:
    r = break_even_analysis(*params)
    print(f'{scenario}')
    print(f'  No cache: ${r["no_cache_cost"]:>8.2f} | With cache: ${r["with_cache_cost"]:>8.2f}')
    print(f'  Savings: ${r["savings"]:.2f} ({r["savings_pct"]}%) | Break-even: {r["break_even_qph"]} qph')
    print()


## Cell 7: Versioned Cache Invalidation


In [ ]:
# Auto-invalidate when documents change
def get_or_create_versioned_cache(
    doc_uris: list[str],
    system_instruction: str,
    prefix: str = 'documind-corpus'
):
    '''Returns cache_name. Creates new only if content changed.'''
    # Hash the doc set for version tracking
    content_key = '|'.join(sorted(doc_uris))
    version = hashlib.sha256(content_key.encode()).hexdigest()[:8]
    display_name = f'{prefix}-{version}'
    
    # Check if this version exists
    for cache in client.caches.list():
        if cache.display_name == display_name:
            print(f'Reusing existing cache: {display_name}')
            return cache.name
    
    # Delete older versions with same prefix
    for cache in client.caches.list():
        if cache.display_name and cache.display_name.startswith(f'{prefix}-'):
            try:
                client.caches.delete(name=cache.name)
                print(f'Deleted old version: {cache.display_name}')
            except Exception as e:
                print(f'Skip delete: {e}')
    
    # Create new
    new_cache = client.caches.create(
        model='gemini-3.6-flash',
        config=CreateCachedContentConfig(
            display_name=display_name,
            system_instruction=system_instruction,
            contents=[Content(role='user', parts=[
                Part.from_uri(file_uri=uri, mime_type='application/pdf')
                for uri in doc_uris
            ])],
            ttl='14400s',  # 4 hours
        ),
    )
    print(f'Created new cache: {display_name}')
    return new_cache.name

print('Versioned cache pattern ready')
print('Same inputs -> same cache. Changed inputs -> new cache, old deleted.')


## Cell 8: Full DocuMind Tiered Caching Pipeline


In [ ]:
# Production pipeline: shared org cache + per-user caches + implicit
class DocuMindCacheManager:
    def __init__(self, client):
        self.client = client
        self.org_cache_name = None
        self.user_caches = {}  # user_id -> cache_name
    
    def init_org_cache(self, org_docs: list[str]):
        '''Shared explicit cache for organization-wide documents.'''
        self.org_cache_name = get_or_create_versioned_cache(
            doc_uris=org_docs,
            system_instruction='You are DocuMind AI. Use org policies to answer.',
            prefix='documind-org'
        )
        return self.org_cache_name
    
    def get_user_cache(self, user_id: str, user_docs: list[str]):
        '''Per-user explicit cache for personal documents.'''
        if user_docs and user_id not in self.user_caches:
            self.user_caches[user_id] = get_or_create_versioned_cache(
                doc_uris=user_docs,
                system_instruction=f'You are DocuMind AI for user {user_id}.',
                prefix=f'documind-user-{user_id}'
            )
        return self.user_caches.get(user_id)
    
    def answer(self, user_id: str, question: str, scope: str = 'org'):
        '''Route query to appropriate cache.'''
        cache_name = self.org_cache_name if scope == 'org' else self.user_caches.get(user_id)
        
        if not cache_name:
            # Fallback: rely on implicit caching
            response = self.client.models.generate_content(
                model='gemini-3.6-flash',
                contents=question
            )
        else:
            response = self.client.models.generate_content(
                model='gemini-3.6-flash',
                contents=question,
                config=GenerateContentConfig(cached_content=cache_name)
            )
        
        um = response.usage_metadata
        return {
            'answer': response.text,
            'cached_tokens': um.cached_content_token_count or 0,
            'total_input': um.prompt_token_count,
            'scope': scope
        }
    
    def cleanup(self):
        '''Delete all caches this manager created.'''
        if self.org_cache_name:
            try: self.client.caches.delete(name=self.org_cache_name)
            except: pass
        for cn in self.user_caches.values():
            try: self.client.caches.delete(name=cn)
            except: pass

print('DocuMindCacheManager ready')
print('Tiered: org cache (shared) + user caches + implicit fallback')


## Done!
- Implicit caching verification via usage_metadata
- Explicit cache create/list/update TTL/delete
- Multi-turn chat with cached context
- RAG cache pattern (corpus cached, query varies)
- Break-even calculator across scenarios
- Versioned invalidation with SHA256 hashing
- Full DocuMind tiered caching pipeline
